# ER-CyRIS Siklus 2 — Koreksi UNSW dan Uji M0 Terbatas

**Tujuan:** membuktikan pembacaan 49 kolom dan label, menghitung benturan duplikat pada pembagian lama, membuat partisi tanpa grup fitur identik lintas batas, dan menjalankan kontrol XGBoost/Random Forest M0 pada UNSW. **Ini bukan** validasi lengkap M1–M4, H1–H5, q_e, atau gerbang risiko. Arsip eksperimen lama tetap utuh. Gunakan berkas mentah yang sudah tersimpan di `MyDrive/ercyris_siklus2/`; tidak ada kebutuhan mengunggah log besar ke ChatGPT.

Jalankan sel berurutan di Google Colab. Output baru ditulis ke folder `ercyris_siklus2/unsw_corrected_v1/`; JSON ringkasan kecil yang perlu dibagikan bernama `unsw_corrected_decision.json`. Semua nomor baris dan hitungan dipertahankan untuk ketertelusuran, tetapi tidak menyimpan isi mentah dalam JSON. Rancangan ini mengubah pembagian data dan penyandian kategori dari eksperimen lama; angka M0 baru tidak boleh disebut replikasi identik hasil lama.

**Koreksi 26 September 2026:** skema 80/20 dan empat partisi menggunakan sampel 30.000 baris yang sama. Karena M0 dilatih dan diuji pada seluruh sampel itu, kolom `holdout` empat partisi pada manifest hanya skema eksplorasi; **bukan** holdout konfirmatori yang belum pernah tersentuh. JSON keluaran mencatat benturan lintas skema. Siapkan holdout baru dari baris yang tidak terpakai sebelum eksperimen konfirmatori.

In [ ]:
import csv, json, sys, platform, hashlib
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier=None

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/ercyris_siklus2')
except ImportError:
    BASE=Path('.')  # Pengujian lokal; tetap wajib sumber CSV pengguna.
P1=BASE/'UNSW-NB15_1.csv'
P2=BASE/'UNSW-NB15_2.csv'
OUT=BASE/'unsw_corrected_v1'
OUT.mkdir(parents=True,exist_ok=True)
SEED=42
CAP=30_000
RUN_M0=True   # Set False jika hanya ingin audit tanpa pelatihan.
assert P1.is_file(), f'Tidak ditemukan: {P1}'
print('Sumber:',P1,'| File 2 tersedia:',P2.is_file(),'| Output:',OUT)

## 1. Baca 49 kolom dan validasi label

Skema mengikuti daftar kolom yang telah digunakan notebook Siklus 1. Nilai label tidak dipaksa menjadi 0 saat tidak terbaca: eksperimen harus berhenti jika skema atau label tidak sah.

In [ ]:
COLS=['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
      'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
      'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
      'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
      'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
      'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
      'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
      'ct_dst_src_ltm','attack_cat','label']
EXCLUDE=['label','attack_cat','srcip','dstip','stime','ltime']
FEATURES=[c for c in COLS if c not in EXCLUDE]
CATS=['proto','state','service']
NUMS=[c for c in FEATURES if c not in CATS]
assert len(COLS)==49 and len(FEATURES)==43
with P1.open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
    first=next(csv.reader(f))
if len(first)!=49 or first[-1].strip() not in ('0','1'):
    raise ValueError('Berkas pertama tidak sesuai skema tanpa header 49 kolom + label biner; STOP.')
raw=pd.read_csv(P1,header=None,names=COLS,low_memory=False)
if raw.shape[1]!=49: raise ValueError('Jumlah kolom berubah; STOP.')
labels=pd.to_numeric(raw['label'],errors='coerce')
if labels.isna().any() or not labels.isin([0,1]).all():
    raise ValueError(f'Label invalid={int((~labels.isin([0,1])).sum())}; STOP.')
y=labels.to_numpy(dtype=np.int8)
if set(np.unique(y))!={0,1}: raise ValueError('Tidak ada kedua kelas pada berkas; STOP.')
assert len(raw)==len(y)
print(f'Data={len(raw):,}; fitur substantif={len(FEATURES)}; positif={int(y.sum()):,}.')
print('Baris awal sumber tidak lagi hilang. Jenis fitur:', {'numeric':len(NUMS),'categorical':len(CATS)})

## 2. Ukur duplikat mentah, grup fitur, dan nilai hilang sebenarnya

Duplikat seluruh 49 kolom dan grup dengan 43 fitur model dihitung terpisah. Dua baris dapat berbeda pada IP/waktu tetapi memiliki fitur model identik; karena itu pemeriksaan grup memakai fitur yang *akan masuk model*. Bila grup yang sama memuat dua label, kedua label tetap dicatat dan grup harus berada pada partisi yang sama.

In [ ]:
raw_duplicate=int(raw.duplicated(subset=COLS,keep='first').sum())
# Hash konsisten atas kolom model. Benturan hash 64-bit praktis sangat jarang;
# angka ini merupakan penapisan kesamaan fitur, bukan bukti independensi sesi/host/waktu.
group_series=pd.util.hash_pandas_object(raw[FEATURES],index=False)
groups=group_series.to_numpy(dtype=np.uint64)
group_count=int(pd.Series(groups).nunique())
conflicts=int(pd.DataFrame({'g':groups,'y':y}).groupby('g')['y'].nunique().gt(1).sum())
numeric=raw[NUMS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
num_invalid=numeric.isna().sum().sort_values(ascending=False)
cat_null={c:int(raw[c].isna().sum()) for c in CATS}
# Menilai fitur konstan sebelum pelatihan tanpa mengganti missing dengan nol.
all_nan=[str(c) for c in num_invalid.index if int(num_invalid[c])==len(raw)]
if all_nan: raise ValueError(f'Fitur numerik seluruhnya kosong setelah parser: {all_nan}; STOP.')
base_audit={'source_file':P1.name,'source_size_bytes':P1.stat().st_size,'records':int(len(raw)),
            'binary_positives':int(y.sum()),'feature_count':len(FEATURES),
            'raw_exact_duplicate_occurrences':raw_duplicate,'unique_model_feature_groups':group_count,
            'groups_with_conflicting_labels':conflicts,
            'numeric_missing_top10':{str(k):int(v) for k,v in num_invalid.head(10).items()},
            'categorical_null':cat_null,'all_nan_numeric_features':all_nan}
print(json.dumps(base_audit,indent=2,ensure_ascii=False))

## 3. Ukur kebocoran pada *pembagian lama* (tanpa menjalankan model lama)

Notebook benchmark dan gerbang terdahulu mengambil 30.000 baris dari hasil `pd.read_csv` yang menghilangkan baris pertama, lalu memakai pembagian stratifikasi 55/15/15/15. Notebook ablasi memakai split 80/20 pada seluruh berkas. Hitungan di bawah merekonstruksi indeks tersebut; ia mengukur kesamaan fitur lintas partisi. Ia **tidak** membuktikan secara retrospektif bahwa semua artefak yang pernah dijalankan memakai berkas dan seed yang sama.

In [ ]:
def legacy_four_way(labels, seed=42):
    index=np.arange(len(labels),dtype=np.int32)
    train,rest=train_test_split(index,train_size=.55,random_state=seed,stratify=labels)
    middle,holdout=train_test_split(rest,test_size=1/3,random_state=seed+1,stratify=labels[rest])
    cal,dev=train_test_split(middle,test_size=.5,random_state=seed+2,stratify=labels[middle])
    return {'train':train,'calibration':cal,'development':dev,'holdout':holdout}

def overlap_matrix(partition,group_ids):
    names=list(partition); sets={k:set(map(int,group_ids[v])) for k,v in partition.items()}
    out={}
    for i,a in enumerate(names):
        for b in names[i+1:]:
            out[f'{a}__{b}']=int(len(sets[a]&sets[b]))
    return out

legacy_group=groups[1:]
legacy_y=y[1:]
cap_old,_=train_test_split(np.arange(len(legacy_y),dtype=np.int32),
                            train_size=CAP,random_state=42,stratify=legacy_y)
cap_old=np.sort(cap_old)
legacy_parts=legacy_four_way(legacy_y[cap_old],SEED)
legacy_4_overlap=overlap_matrix(legacy_parts,legacy_group[cap_old])
old_train,old_test=train_test_split(np.arange(len(legacy_y),dtype=np.int32),
                                    test_size=.20,random_state=SEED,stratify=legacy_y)
legacy_2_overlap=overlap_matrix({'train':old_train,'test':old_test},legacy_group)
legacy={'sample_cap_30k':int(len(cap_old)),
        'cap_duplicate_occurrences_by_model_features':int(pd.Series(legacy_group[cap_old]).duplicated().sum()),
        'four_way_shared_feature_groups':legacy_4_overlap,
        'all_data_80_20_shared_feature_groups':legacy_2_overlap,
        'limitation':'Historical index reconstruction; not an executed historical output manifest.'}
print(json.dumps(legacy,indent=2,ensure_ascii=False))

## 4. Bentuk partisi baru tanpa grup fitur identik lintas batas

Sampling 30.000 pertama dilakukan dengan label untuk cakupan kelas; hasilnya kemudian dipecah memakai grup fitur. Ini mengubah komposisi dan definisi holdout dibanding hasil lama. Indeks dan SHA-256 berkas mentah dicatat; **jangan** gunakan registry ambang lama untuk partisi baru. Partisi 80/20 dipakai hanya untuk kontrol M0; partisi 55/15/15/15 disiapkan untuk uji siklus berikutnya.

In [ ]:
def gsplit(index, frac, seed):
    splitter=GroupShuffleSplit(n_splits=1,train_size=frac,random_state=seed)
    a,b=next(splitter.split(index,y[index],groups[index]))
    return index[a],index[b]

cap,_=train_test_split(np.arange(len(y),dtype=np.int32),train_size=CAP,
                        random_state=SEED,stratify=y)
cap=np.sort(cap)
tr2,te2=gsplit(cap,.8,SEED)
tr4,remainder=gsplit(cap,.55,SEED)
mid4,ho4=gsplit(remainder,2/3,SEED+1)
cal4,dev4=gsplit(mid4,.5,SEED+2)
new2={'train':np.sort(tr2),'test':np.sort(te2)}
new4={'train':np.sort(tr4),'calibration':np.sort(cal4),
      'development':np.sort(dev4),'holdout':np.sort(ho4)}
for partition in (new2,new4):
    assert sum(map(len,partition.values()))==CAP
    assert len(set(np.concatenate(list(partition.values()))))==CAP
    overlap=overlap_matrix(partition,groups)
    if any(overlap.values()): raise AssertionError(f'Grup fitur melintasi partisi: {overlap}')
    if any(y[ix].sum()==0 or y[ix].sum()==len(ix) for ix in partition.values()):
        raise ValueError('Partisi satu kelas: ganti seed dan catat protokol sebelum menjalankan model.')

def part_counts(parts):
    return {k:{'n':int(len(ix)),'positive':int(y[ix].sum()),
               'positive_fraction':float(y[ix].mean()),
               'feature_groups':int(pd.Series(groups[ix]).nunique())}
            for k,ix in parts.items()}
# Seluruh 30.000 sampel pada skema 80/20 telah menjadi train ATAU test M0.
# Hitungan lintas skema ini mencegah salah menyebut holdout empat partisi 'untouched'.
cross_scheme={f'{a}__{b}':int(len(set(map(int,idx)) & set(map(int,jdx))))
              for a,idx in new2.items() for b,jdx in new4.items()}
assert sum(cross_scheme[f'{a}__holdout'] for a in new2)==len(new4['holdout'])
print('PERINGATAN: holdout empat partisi telah terpapar skema 80/20:',cross_scheme)

partitions={'two_way':part_counts(new2),'four_way':part_counts(new4),
            'two_way_overlap':overlap_matrix(new2,groups),
            'four_way_overlap':overlap_matrix(new4,groups)}
print(json.dumps(partitions,indent=2))
# Isi data mentah tidak diekspor. Indeks dicatat agar eksekusi ulang bisa ditelusuri.
rows=[]
for scheme,parts in [('80_20',new2),('55_15_15_15',new4)]:
    for name,idx in parts.items():
        rows.extend((scheme,name,int(i),int(y[i]),str(int(groups[i]))) for i in idx)
pd.DataFrame(rows,columns=['scheme','partition','source_row_zero_based','label','feature_group_hash64'])  .to_csv(OUT/'unsw_corrected_split_manifest.csv',index=False)
print('Manifest dibuat:',OUT/'unsw_corrected_split_manifest.csv')

## 5. Kontrol M0 setelah koreksi data

Praproses numerik (median dan MinMax) serta OneHot kategori *hanya di-fit pada training*. Nilai yang hilang tidak diganti global sebelum split. Metrik `AP` dihitung dengan `average_precision_score`; nama kolom arsip lama `PR_AUC` tidak digunakan sebagai nama metrik baru. Noise σ=0,05 diterapkan pada matriks fitur hasil transformasi untuk pemeriksaan operasional terbatas. Ini bukan ukuran ketahanan host/waktu, risiko, atau M1–M4.

In [ ]:
def transform_train_only(frame,train_idx,test_idx):
    work=frame[FEATURES].copy()
    work[NUMS]=work[NUMS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in CATS:
        work[c]=work[c].astype('string').fillna('__MISSING__').astype(str)
    try:
        enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32)
    except TypeError:
        enc=OneHotEncoder(handle_unknown='ignore',sparse=False,dtype=np.float32)
    prep=ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),
                             ('scale',MinMaxScaler())]),NUMS),
        ('category',enc,CATS)
    ],remainder='drop',sparse_threshold=0)
    xtrain=np.asarray(prep.fit_transform(work.iloc[train_idx]),dtype=np.float32)
    xtest=np.asarray(prep.transform(work.iloc[test_idx]),dtype=np.float32)
    if xtrain.shape[1]!=xtest.shape[1] or not np.isfinite(xtrain).all() or not np.isfinite(xtest).all():
        raise ValueError('Transformasi gagal: dimensi atau nilai nonhingga; STOP.')
    return xtrain,xtest,prep

def metrics(ytrue,prob,threshold=.5):
    pred=(prob>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(ytrue,pred,labels=[0,1]).ravel()
    return {'F1':float(f1_score(ytrue,pred,zero_division=0)),
            'Precision':float(precision_score(ytrue,pred,zero_division=0)),
            'Recall':float(recall_score(ytrue,pred,zero_division=0)),
            'AP':float(average_precision_score(ytrue,prob)),
            'FAR':float(fp/(fp+tn)) if (fp+tn) else None,
            'Alert_Rate':float(pred.mean()),
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)}

m0=[]
if RUN_M0:
    if XGBClassifier is None:
        raise ImportError('XGBoost belum terpasang di Colab. Jalankan %pip install xgboost, lalu ulangi notebook dari awal.')
    xtrain,xtest,processor=transform_train_only(raw,new2['train'],new2['test'])
    print('Train-only transform:',xtrain.shape,xtest.shape)
    models={
        'XGBoost':XGBClassifier(random_state=SEED,n_jobs=-1,eval_metric='logloss',
                                verbosity=0,tree_method='hist'),
        'RandomForest':RandomForestClassifier(random_state=SEED,n_jobs=-1),
    }
    for name,model in models.items():
        model.fit(xtrain,y[new2['train']])
        p0=model.predict_proba(xtest)[:,1]
        rng=np.random.RandomState(SEED)
        noisy=np.clip(xtest+rng.normal(0,.05,xtest.shape),0,1).astype(np.float32)
        pn=model.predict_proba(noisy)[:,1]
        m0.append({'model':name,'clean':metrics(y[new2['test']],p0),
                   'noise_sigma_0p05':metrics(y[new2['test']],pn),
                   'threshold':.5,'positive_class':1,
                   'note':'Pemeriksaan M0 baru; protokol kategori/partisi berbeda dari hasil lama.'})
        print(name,'F1',round(m0[-1]['clean']['F1'],4),'AP',round(m0[-1]['clean']['AP'],4),
              'F1_noise',round(m0[-1]['noise_sigma_0p05']['F1'],4))
else:
    print('Audit saja: pelatihan M0 dilewati.')

## 6. Simpan keputusan dan keluaran yang perlu dikirim

File2 **tidak** otomatis menjadi holdout mandiri. Hasil audit skema sebelumnya mencatat satu baris identik lintas berkas, tetapi kesamaan host, aliran, waktu, atau fitur belum terukur di sini. Tinjau ringkasan `unsw_corrected_decision.json`, dan kirim hanya JSON itu. File `unsw_corrected_split_manifest.csv` tetap di Drive sebagai arsip replikasi.

In [ ]:
def sha256_file(path,block=4*1024*1024):
    digest=hashlib.sha256()
    with path.open('rb') as fh:
        for piece in iter(lambda:fh.read(block),b''):
            digest.update(piece)
    return digest.hexdigest()

result={'status':'CORRECTED_UNSW_AUDIT_AND_M0_ONLY' if RUN_M0 else 'CORRECTED_UNSW_AUDIT_ONLY',
        'source_sha256':sha256_file(P1),
        'versions':{'python':platform.python_version(),'numpy':np.__version__,
                    'pandas':pd.__version__, 'sklearn':__import__('sklearn').__version__,
                    'xgboost':(__import__('xgboost').__version__ if XGBClassifier is not None else None)},
        'config':{'seed':SEED,'cap':CAP,'source_file_2_used_as_holdout':False,
                  'group_key':'pandas_hash_pandas_object of 43 model features',
                  'old_four_way':'55/15/15/15 stratified random','old_two_way':'80/20 stratified random',
                  'new_four_way':'group split approximately 55/15/15/15',
                  'new_two_way':'group split approximately 80/20',
                  'categorical':'train-only one-hot; unseen category ignored',
                  'numeric':'train-only median imputation and MinMaxScaler',
                  'threshold_m0':0.5,'noise_sigma':0.05},
        'data':base_audit,'historical_split_reconstruction':legacy,
        'corrected_partitions':partitions,'cross_scheme_touch':cross_scheme,
        'four_way_holdout_status':'EXPOSED_TO_80_20_M0_NOT_FINAL_HOLDOUT',
        'm0_control':m0,
        'remaining_unvalidated':['M1 architecture','M2','M3','M4','H1-H5 pooled',
                                 'six-family diversity rerun','frozen gate rerun',
                                 'q_e decision rule','host/time independent holdout'],
        'limitations':['No full framework validation.','No external independent evaluation on source file 2.',
                       'Four-way holdout reused by 80/20 M0 and cannot serve as untouched final holdout.',
                       'Reconstructed old splits presume the same source file and seeds.',
                       '64-bit group hashes screen exact predictor equality; not host/time independence.']}
TARGET=OUT/'unsw_corrected_decision.json'
TARGET.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print('Simpan dan kirim:',TARGET)
print('Status:',result['status'])